# AS/400 Phase 1 – Systemanalyse
### Auswertung der Systemkatalog-Extraktion

**Voraussetzung:** `as400_phase1_extract.py` wurde erfolgreich ausgeführt und hat eine `as400_phase1_YYYYMMDD_HHMM.xlsx` erzeugt.

```
pip install pandas matplotlib seaborn openpyxl
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import glob
import warnings
warnings.filterwarnings('ignore')

# Finde die aktuellste Extraktionsdatei automatisch
files = sorted(glob.glob('as400_phase1_*.xlsx'), reverse=True)
if not files:
    raise FileNotFoundError('Keine as400_phase1_*.xlsx gefunden. Bitte zuerst Extract-Script ausführen.')
EXCEL_FILE = files[0]
print(f'Lade: {EXCEL_FILE}')

# Alle Sheets laden
sheets = pd.read_excel(EXCEL_FILE, sheet_name=None)
print(f'Verfügbare Sheets: {list(sheets.keys())}')

---
## 1 · System-Überblick

In [ ]:
schemas = sheets.get('00_Übersicht_Schemas', pd.DataFrame())
tabellen = sheets.get('01_Tabellen', pd.DataFrame())
views = sheets.get('03_Views', pd.DataFrame())
indizes = sheets.get('04_Indizes', pd.DataFrame())
programme = sheets.get('08_Programme', pd.DataFrame())

print('=' * 50)
print('  AS/400 SYSTEM-SNAPSHOT')
print('=' * 50)
print(f'  Schemas / Libraries  : {len(schemas):>6,}')
print(f'  Physische Tabellen   : {len(tabellen):>6,}')
print(f'  Views                : {len(views):>6,}')
print(f'  Indizes              : {len(indizes):>6,}')
print(f'  Programme/Routinen   : {len(programme):>6,}')
if 'daten_mb' in tabellen.columns:
    gesamt_mb = tabellen['daten_mb'].sum()
    print(f'  Datenmenge gesamt    : {gesamt_mb:>6,.1f} MB')
print('=' * 50)

---
## 2 · Schemas / Libraries im Überblick

In [ ]:
if not schemas.empty:
    top = schemas.sort_values('anzahl_tabellen', ascending=False).head(20)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Schemas / Libraries – Übersicht', fontsize=14, fontweight='bold', y=1.01)

    # Balken: Anzahl Tabellen pro Schema
    colors = ['#1F4E79' if i == 0 else '#2E75B6' if i < 3 else '#9DC3E6' for i in range(len(top))]
    axes[0].barh(top['schema_name'], top['anzahl_tabellen'], color=colors)
    axes[0].set_xlabel('Anzahl Tabellen')
    axes[0].set_title('Top-20 Schemas nach Tabellenanzahl')
    axes[0].invert_yaxis()
    for i, (val, name) in enumerate(zip(top['anzahl_tabellen'], top['schema_name'])):
        axes[0].text(val + 0.5, i, str(val), va='center', fontsize=9)

    # Torte: Tabellen vs. Views vs. Aliases
    totals = schemas[['tabellen', 'views', 'aliases']].sum()
    wedge_colors = ['#1F4E79', '#2E75B6', '#9DC3E6']
    axes[1].pie(totals, labels=['Tabellen', 'Views', 'Aliases'],
                colors=wedge_colors, autopct='%1.1f%%', startangle=90,
                textprops={'fontsize': 11})
    axes[1].set_title('Verteilung Objekt-Typen')

    plt.tight_layout()
    plt.savefig('chart_01_schemas.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nGespeichert: chart_01_schemas.png')

---
## 3 · Größte Tabellen (nach Zeilenzahl)

In [ ]:
stat = sheets.get('06_Tabellenstatistik', pd.DataFrame())

if not stat.empty and 'zeilen_anzahl' in stat.columns:
    top50 = stat.sort_values('zeilen_anzahl', ascending=False).head(50).copy()
    top50['label'] = top50['schema_name'] + '.' + top50['tabelle']
    top20 = top50.head(20)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Tabellen nach Datenmenge', fontsize=14, fontweight='bold')

    # Balken: Top 20 nach Zeilen
    bar_colors = ['#C00000' if v > top20['zeilen_anzahl'].quantile(0.9)
                  else '#2E75B6' for v in top20['zeilen_anzahl']]
    axes[0].barh(top20['label'], top20['zeilen_anzahl'], color=bar_colors)
    axes[0].set_xlabel('Anzahl Zeilen')
    axes[0].set_title('Top-20 Tabellen nach Zeilenzahl')
    axes[0].invert_yaxis()
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    # Scatter: Zeilen vs. MB
    if 'daten_mb' in stat.columns:
        axes[1].scatter(top50['zeilen_anzahl'], top50['daten_mb'],
                        c='#2E75B6', alpha=0.7, edgecolors='white', linewidth=0.5, s=60)
        # Top-5 beschriften
        for _, row in top50.head(5).iterrows():
            axes[1].annotate(row['tabelle'], (row['zeilen_anzahl'], row['daten_mb']),
                             textcoords='offset points', xytext=(6, 4), fontsize=8)
        axes[1].set_xlabel('Zeilen')
        axes[1].set_ylabel('Datengröße (MB)')
        axes[1].set_title('Zeilen vs. MB (Top-50)')
        axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    plt.tight_layout()
    plt.savefig('chart_02_groesse.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gespeichert: chart_02_groesse.png')
    print(f'\nTop-10 Tabellen:')
    display(top20[['label','zeilen_anzahl','daten_mb']].head(10).reset_index(drop=True))

---
## 4 · Spalten-Analyse

In [ ]:
cols = sheets.get('02_Spalten', pd.DataFrame())

if not cols.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Spalten-Analyse', fontsize=14, fontweight='bold')

    # Datentypen
    dtype_counts = cols['datentyp'].value_counts().head(12)
    axes[0].bar(range(len(dtype_counts)), dtype_counts.values, color='#2E75B6')
    axes[0].set_xticks(range(len(dtype_counts)))
    axes[0].set_xticklabels(dtype_counts.index, rotation=45, ha='right', fontsize=9)
    axes[0].set_title('Häufigste Datentypen')
    axes[0].set_ylabel('Anzahl Spalten')

    # Nullable-Anteil
    if 'nullable' in cols.columns:
        null_counts = cols['nullable'].value_counts()
        axes[1].pie(null_counts, labels=[f'Nullable ({v})' if k == 'Y' else f'NOT NULL ({v})'
                                          for k, v in null_counts.items()],
                    colors=['#9DC3E6', '#1F4E79'], autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Nullable vs. NOT NULL')

    # Spaltenanzahl pro Tabelle (Histogramm)
    col_per_table = cols.groupby(['schema_name', 'tabelle']).size()
    axes[2].hist(col_per_table, bins=30, color='#2E75B6', edgecolor='white')
    axes[2].axvline(col_per_table.median(), color='#C00000', linestyle='--',
                    label=f'Median: {col_per_table.median():.0f}')
    axes[2].set_xlabel('Spalten pro Tabelle')
    axes[2].set_ylabel('Anzahl Tabellen')
    axes[2].set_title('Verteilung: Spalten pro Tabelle')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('chart_03_spalten.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gespeichert: chart_03_spalten.png')

    # Tabellen mit besonders vielen Spalten (mögliche Gott-Tabellen)
    fat = col_per_table[col_per_table > col_per_table.quantile(0.9)].reset_index()
    fat.columns = ['schema', 'tabelle', 'spalten_anzahl']
    print(f'\n⚠ Tabellen mit überdurchschnittlich vielen Spalten (Top-Quantil, {len(fat)} Stück):')
    display(fat.sort_values('spalten_anzahl', ascending=False).head(20).reset_index(drop=True))

---
## 5 · Index-Profil (Hinweise auf Kern-Tabellen)

In [ ]:
idx = sheets.get('04_Indizes', pd.DataFrame())

if not idx.empty:
    idx_per_table = idx.groupby(['schema_name', 'tabelle']).size().reset_index(name='index_anzahl')
    unique_idx = idx[idx['eindeutig'] == 'Y'].groupby(['schema_name', 'tabelle']).size().reset_index(name='unique_indizes')
    idx_summary = idx_per_table.merge(unique_idx, on=['schema_name', 'tabelle'], how='left').fillna(0)
    idx_summary['unique_indizes'] = idx_summary['unique_indizes'].astype(int)

    top_idx = idx_summary.sort_values('index_anzahl', ascending=False).head(25)
    top_idx['label'] = top_idx['schema_name'] + '.' + top_idx['tabelle']

    fig, ax = plt.subplots(figsize=(14, 7))
    x = range(len(top_idx))
    ax.bar(x, top_idx['index_anzahl'], label='Indizes gesamt', color='#2E75B6')
    ax.bar(x, top_idx['unique_indizes'], label='Unique-Indizes', color='#1F4E79')
    ax.set_xticks(x)
    ax.set_xticklabels(top_idx['label'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Anzahl Indizes')
    ax.set_title('Top-25 Tabellen nach Index-Anzahl\n(Viele Indizes = häufig genutzter Kern-Kandidat)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('chart_04_indizes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gespeichert: chart_04_indizes.png')

    tabellen_ohne_index = len(tabellen) - len(idx_per_table)
    print(f'\nTabellen ohne einen einzigen Index: ~{tabellen_ohne_index}')
    print('→ Auf alten AS/400-Systemen häufig: logische Schlüssel in Programm-Code versteckt!')

---
## 6 · Heatmap – Aktivitätsprofil
Kombiniert Zeilenzahl + Indexanzahl + Spaltenanzahl zu einem "Wichtigkeits-Score"

In [ ]:
if not tabellen.empty and not stat.empty and not idx.empty:
    # Score-Berechnung
    t = tabellen[['schema_name', 'tabelle', 'spalten_anzahl']].copy()
    s = stat[['schema_name', 'tabelle', 'zeilen_anzahl', 'daten_mb']].copy()
    i = idx.groupby(['schema_name', 'tabelle']).size().reset_index(name='index_anzahl')

    merged = t.merge(s, on=['schema_name', 'tabelle'], how='left')
    merged = merged.merge(i, on=['schema_name', 'tabelle'], how='left').fillna(0)

    # Normalisierung 0-1 und gewichteter Score
    for col in ['zeilen_anzahl', 'spalten_anzahl', 'index_anzahl']:
        mx = merged[col].max()
        merged[f'{col}_norm'] = merged[col] / mx if mx > 0 else 0

    merged['wichtigkeit_score'] = (
        merged['zeilen_anzahl_norm']  * 0.5 +
        merged['index_anzahl_norm']   * 0.3 +
        merged['spalten_anzahl_norm'] * 0.2
    )

    top40 = merged.sort_values('wichtigkeit_score', ascending=False).head(40).copy()
    top40['label'] = top40['schema_name'] + '.' + top40['tabelle']

    heat_data = top40.set_index('label')[['zeilen_anzahl_norm', 'spalten_anzahl_norm', 'index_anzahl_norm']]
    heat_data.columns = ['Zeilen (norm.)', 'Spalten (norm.)', 'Indizes (norm.)']

    fig, ax = plt.subplots(figsize=(10, 14))
    sns.heatmap(heat_data, cmap='Blues', ax=ax, linewidths=0.3,
                cbar_kws={'label': 'Normierter Wert (0=min, 1=max)'})
    ax.set_title('Aktivitätsprofil – Top-40 Tabellen\n(Kandidaten für Kern-Entitäten)', fontsize=13)
    ax.set_xlabel('')
    plt.tight_layout()
    plt.savefig('chart_05_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gespeichert: chart_05_heatmap.png')
    print(f'\nTop-10 Tabellen nach Wichtigkeits-Score:')
    display(top40[['label', 'zeilen_anzahl', 'spalten_anzahl', 'index_anzahl', 'wichtigkeit_score']]
            .head(10).reset_index(drop=True))

---
## 7 · Beschreibungsqualität (Fehlende Dokumentation)
Auf AS/400 sind Beschreibungsfelder oft leer – das ist ein Risiko-Indikator.

In [ ]:
if not tabellen.empty and 'beschreibung' in tabellen.columns:
    described = tabellen['beschreibung'].notna() & (tabellen['beschreibung'].str.strip() != '')
    pct = described.mean() * 100

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Dokumentationsqualität', fontsize=14, fontweight='bold')

    # Gesamt
    axes[0].pie([described.sum(), (~described).sum()],
                labels=[f'Mit Beschreibung ({described.sum()})',
                        f'Ohne Beschreibung ({(~described).sum()})'],
                colors=['#00B050', '#C00000'], autopct='%1.1f%%', startangle=90)
    axes[0].set_title(f'Tabellen mit Beschreibung\nGesamt: {pct:.1f}%')

    # Pro Schema
    if 'schema_name' in tabellen.columns:
        qual = tabellen.groupby('schema_name').apply(
            lambda g: (g['beschreibung'].notna() & (g['beschreibung'].str.strip() != '')).mean() * 100
        ).sort_values()
        color_list = ['#C00000' if v < 30 else '#FF8C00' if v < 70 else '#00B050' for v in qual]
        axes[1].barh(qual.index, qual.values, color=color_list)
        axes[1].set_xlabel('% Tabellen mit Beschreibung')
        axes[1].set_title('Dokumentationsquote pro Schema')
        axes[1].axvline(50, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig('chart_06_doku.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Gespeichert: chart_06_doku.png')

    if pct < 50:
        print(f'\n⚠ Nur {pct:.1f}% der Tabellen haben eine Beschreibung.')
        print('→ Starkes Indiz für Wissensmonopol – dieses Wissen liegt nur in Köpfen.')

---
## 8 · Zusammenfassung für Weiterarbeit

In [ ]:
print('=== ZUSAMMENFASSUNG PHASE 1 ===')
print()
if not tabellen.empty:
    print(f'System-Inventar:')
    print(f'  {len(tabellen):>5} physische Tabellen')
    print(f'  {len(views):>5} Views')
    print(f'  {len(schemas):>5} Schemas/Libraries')
    print()

if not stat.empty and 'zeilen_anzahl' in stat.columns:
    print(f'Daten-Profil:')
    print(f'  Gesamtzeilen  : {stat["zeilen_anzahl"].sum():>12,.0f}')
    if 'daten_mb' in stat.columns:
        print(f'  Gesamt-Größe  : {stat["daten_mb"].sum():>12,.1f} MB')
    leer = len(tabellen) - len(stat[stat['zeilen_anzahl'] > 0])
    print(f'  Leere Tabellen: {leer:>12}')
    print()

print('Nächste Schritte:')
print('  → Phase 2: DSPRGMREF – Programmreferenzen und Call-Graphen')
print('  → Kern-Entitäten aus Heatmap mit den bekannten ~90 Tabellen abgleichen')
print('  → Tabellen ohne Index und ohne Beschreibung dokumentieren')